# 5.2 — 誤解を生まない比較


同じ原資料でも、総量・一人当たり・割合は別の問いへ答えます。先に指標と粒度を確定し、比較表を表示してから描画します。


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

project = Path.cwd() / "projects" / "clinic-wait-evidence"
records = pd.read_csv(project / "data" / "clinic-waits-practice.csv")


## 5.2.1 互換性のある値を先に合計する


In [ ]:
service = records.groupby(["clinic_id", "clinic_name", "time_slot"], as_index=False).agg(
    records=("week", "size"),
    patients_seen=("patients_seen", "sum"),
    total_wait_minutes=("total_wait_minutes", "sum"),
    over_60_minutes=("over_60_minutes", "sum"),
)
service["average_wait_minutes"] = service["total_wait_minutes"] / service["patients_seen"]
service["over_60_rate"] = service["over_60_minutes"] / service["patients_seen"] * 100
display(service)
assert service["patients_seen"].sum() == records["patients_seen"].sum()


## 5.2.2 総負担と個人の経験を分ける


In [ ]:
burden = records.groupby(["clinic_id", "clinic_name"], as_index=False).agg(
    total_wait_minutes=("total_wait_minutes", "sum"),
    patients_seen=("patients_seen", "sum"),
)
burden["average_wait_minutes"] = burden["total_wait_minutes"] / burden["patients_seen"]
print("Largest total burden")
display(burden.sort_values("total_wait_minutes", ascending=False).head(1))
print("Longest average wait by clinic and time slot")
display(service.sort_values("average_wait_minutes", ascending=False).head(1))


## 5.2.3 軸・順序・件数を比較条件として示す


In [ ]:
plot_data = service.assign(service=service["clinic_name"] + " — " + service["time_slot"]).sort_values("average_wait_minutes")
ax = plot_data.plot.barh(x="service", y="average_wait_minutes", legend=False)
ax.set(title="Average wait by clinic and time slot", xlabel="Average wait per patient (minutes)", ylabel="")
ax.set_xlim(left=0)
for index, row in enumerate(plot_data.itertuples()):
    ax.text(row.average_wait_minutes + .5, index, f"n={row.patients_seen}", va="center")
plt.tight_layout()
plt.show()


## 統合練習
診療所別の60分超人数合計と、診療所・時間帯別の60分超割合を作り、二つの順位が違う理由を説明してください。
